# passlib

Smoke test for [passlib](https://pypi.org/project/passlib/): hashing and verifying with several
schemes, known-answer vectors cross-checked against `hashlib`, `CryptContext` policies and hash
migration, the preset application contexts, and TOTP one-time passwords. The bcrypt and argon2
schemes are left out because they depend on separate backend packages. Passwords are generated at
run time or are short dummy values.

## Example 1: Hash and verify with PBKDF2-SHA256, SHA-256-crypt and SHA-512-crypt

In [ ]:
import secrets

import passlib
from passlib.hash import pbkdf2_sha256, sha256_crypt, sha512_crypt

print(f"passlib {passlib.__version__}")

pw = secrets.token_urlsafe(16)
for handler, prefix in ((pbkdf2_sha256, "$pbkdf2-sha256$"), (sha256_crypt, "$5$"), (sha512_crypt, "$6$")):
    hashed = handler.hash(pw)
    assert hashed.startswith(prefix), (handler.name, hashed[:20])
    assert handler.identify(hashed), handler.name
    assert handler.verify(pw, hashed), f"{handler.name} rejected the right password"
    assert not handler.verify(pw + "x", hashed), f"{handler.name} accepted a wrong password"
    print(f"{handler.name:14s} {hashed[:45]}...")

# Settings such as the round count are stored in the hash and parsed back by from_string().
custom = pbkdf2_sha256.using(rounds=1000).hash(pw)
assert pbkdf2_sha256.from_string(custom).rounds == 1000
default = pbkdf2_sha256.hash(pw)
assert pbkdf2_sha256.from_string(default).rounds == pbkdf2_sha256.default_rounds
assert not sha256_crypt.identify(custom), "sha256_crypt claimed a pbkdf2_sha256 hash"
print(f"pbkdf2_sha256 default rounds: {pbkdf2_sha256.default_rounds}")

## Example 2: Known-answer vectors and a cross-check against hashlib

In [ ]:
import hashlib

from passlib.hash import des_crypt, pbkdf2_sha256, sha256_crypt, sha512_crypt
from passlib.utils.binary import ab64_decode

# Test vectors from the SHA-crypt specification ("Unix crypt using SHA-256 and SHA-512"):
# "Hello world!" with the salt "saltstring" and the default 5000 rounds (the same strings
# glibc's crypt(3) produces).
SHA256_VECTOR = "$5$saltstring$5B8vYYiY.CVt1RlTTf8KbXBH3hsxY/GNooZaBBGWEc5"
SHA512_VECTOR = (
    "$6$saltstring$svn8UoSVapNtMuq1ukKS4tPQd8iKwSMHWjl/O817G3uBnIFNjnQJuesI68u4OTLiBFdcbYEdFCoEOfaS35inz1"
)
assert sha256_crypt.verify("Hello world!", SHA256_VECTOR)
assert sha512_crypt.verify("Hello world!", SHA512_VECTOR)
assert sha256_crypt.using(salt="saltstring", rounds=5000).hash("Hello world!") == SHA256_VECTOR
assert sha512_crypt.using(salt="saltstring", rounds=5000).hash("Hello world!") == SHA512_VECTOR
assert not sha256_crypt.verify("Hello world?", SHA256_VECTOR)
print("SHA-crypt vectors match")

# pbkdf2_sha256 stores salt and checksum in "adapted base64"; decode them and recompute the
# checksum with the standard library.
salt = b"0123456789abcdef"
hashed = pbkdf2_sha256.using(salt=salt, rounds=1000).hash("pw-1")
_, ident, rounds, salt_field, checksum_field = hashed.split("$")
assert ident == "pbkdf2-sha256" and rounds == "1000", hashed
assert ab64_decode(salt_field) == salt
assert ab64_decode(checksum_field) == hashlib.pbkdf2_hmac("sha256", b"pw-1", salt, 1000, 32)
print(f"pbkdf2_sha256 matches hashlib.pbkdf2_hmac: {hashed[:40]}...")

# Legacy DES-crypt only looks at the first 8 characters, one reason it is deprecated.
legacy = des_crypt.hash("pw-12345")
assert len(legacy) == 13, legacy
assert des_crypt.verify("pw-12345", legacy)
assert des_crypt.verify("pw-12345-and-more", legacy), "des_crypt did not truncate at 8 characters"
print("des_crypt truncates at 8 characters, as specified")

## Example 3: CryptContext policies and migrating old hashes

In [ ]:
from passlib.context import CryptContext
from passlib.hash import md5_crypt

ctx = CryptContext(
    schemes=["pbkdf2_sha256", "sha256_crypt", "md5_crypt"],
    default="pbkdf2_sha256",
    deprecated=["sha256_crypt", "md5_crypt"],
    pbkdf2_sha256__default_rounds=2000,
)
assert ctx.default_scheme() == "pbkdf2_sha256"
assert ctx.schemes() == ("pbkdf2_sha256", "sha256_crypt", "md5_crypt")

current = ctx.hash("pw-2")
assert ctx.identify(current) == "pbkdf2_sha256"
assert current.startswith("$pbkdf2-sha256$2000$"), current
assert not ctx.needs_update(current)

# A hash made with a deprecated scheme still verifies, but is flagged for replacement.
old = md5_crypt.hash("pw-2")
assert ctx.identify(old) == "md5_crypt"
assert ctx.verify("pw-2", old)
assert ctx.needs_update(old)

# verify_and_update() checks the password and, on success, returns a replacement hash
# in the default scheme. A wrong password gives (False, None).
ok, replacement = ctx.verify_and_update("pw-2", old)
assert ok and replacement is not None
assert ctx.identify(replacement) == "pbkdf2_sha256"
assert ctx.verify_and_update("wrong", old) == (False, None)
assert ctx.verify_and_update("pw-2", current) == (True, None), "an up-to-date hash was replaced"
print("md5_crypt hash migrated to", replacement[:30] + "...")

# Raising the minimum round count marks existing hashes for update.
stricter = ctx.copy(pbkdf2_sha256__min_rounds=5000, pbkdf2_sha256__default_rounds=5000)
assert stricter.needs_update(current)
assert stricter.hash("pw-2").startswith("$pbkdf2-sha256$5000$")

# A policy round-trips through its INI-style serialisation.
restored = CryptContext.from_string(ctx.to_string())
assert restored.to_dict() == ctx.to_dict()
assert restored.verify("pw-2", current)
print(ctx.to_string().strip())

## Example 4: Preset application contexts

In [ ]:
from passlib.apps import custom_app_context, django_context

# custom_app_context: SHA-512-crypt or SHA-256-crypt, suitable for a new application.
app_hash = custom_app_context.hash("pw-3")
assert custom_app_context.identify(app_hash) == custom_app_context.default_scheme()
assert set(custom_app_context.schemes()) >= {"sha256_crypt", "sha512_crypt"}
assert custom_app_context.verify("pw-3", app_hash)
assert not custom_app_context.verify("pw-4", app_hash)
print(f"custom_app_context default: {custom_app_context.default_scheme()}")

# django_context reads and writes the hash format Django stores in its user table.
django_hash = django_context.hash("pw-3")
assert django_context.default_scheme() == "django_pbkdf2_sha256"
assert django_hash.startswith("pbkdf2_sha256$"), django_hash[:20]
algorithm, iterations, salt, checksum = django_hash.split("$")
assert int(iterations) > 0 and salt and checksum
assert django_context.verify("pw-3", django_hash)
assert not django_context.verify("pw-4", django_hash)
print(f"django hash: {algorithm}${iterations}$...")

## Example 5: TOTP one-time passwords

In [ ]:
import base64

from passlib.exc import InvalidTokenError, MalformedTokenError
from passlib.totp import TOTP

# RFC 6238 appendix B test vectors: SHA-1, 8 digits, the ASCII key "12345678901234567890".
rfc_key = base64.b32encode(b"12345678901234567890").decode()
rfc = TOTP(key=rfc_key, digits=8, alg="sha1")
for when, expected in ((59, "94287082"), (1111111109, "07081804"), (1234567890, "89005924"), (2000000000, "69279037")):
    assert rfc.generate(time=when).token == expected, (when, expected)
print("RFC 6238 vectors match")

# A new random key, shared with an authenticator app through an otpauth:// URI.
factory = TOTP.using(issuer="example.org")
totp = factory.new()
uri = totp.to_uri(label="alice@example.org")
assert uri.startswith("otpauth://totp/"), uri[:20]
clone = TOTP.from_uri(uri)
assert clone.base32_key == totp.base32_key
assert clone.issuer == "example.org" and clone.label == "alice@example.org"

# The token belongs to the 30-second time step from 1_699_999_980 to 1_700_000_009.
code = totp.generate(time=1_700_000_000)
assert len(code.token) == 6 and code.token.isdigit(), code.token
match = clone.match(code.token, time=1_700_000_005)
assert match.counter == 1_700_000_000 // 30 and match.skipped == 0, (match.counter, match.skipped)
print(f"token {code.token} accepted within its own 30-second time step")

# match() also accepts a token up to 30 seconds old by default (window=30), to allow for clock
# drift. 10 seconds later is already the next step: accepted by default, rejected with window=0.
late = clone.match(code.token, time=1_700_000_010)
assert late.counter == match.counter and late.skipped == -1, (late.counter, late.skipped)
try:
    clone.match(code.token, time=1_700_000_010, window=0)
except InvalidTokenError:
    print("10 s later: accepted inside the default 30-second window, rejected with window=0")
else:
    raise AssertionError("window=0 accepted a token from the previous time step")

# Outside the window the token is rejected; a token of the wrong length is malformed.
try:
    clone.match(code.token, time=1_700_000_000 + 600)
except InvalidTokenError:
    print("stale token rejected")
else:
    raise AssertionError("a token from 10 minutes earlier was accepted")
try:
    clone.match("12345", time=1_700_000_000)
except MalformedTokenError:
    print("malformed token rejected")
else:
    raise AssertionError("a 5-digit token was accepted")